In [1]:
# CELL 1 — Load staging data
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, date
import hashlib

engine = create_engine(
    "mssql+pyodbc://@LAPTOP-K44AI2HU\\SQLEXPRESS/pharma_db"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

df = pd.read_sql("SELECT * FROM staging.stg_fda_shortages", engine)
print(f"Loaded {len(df)} records from staging")

Loaded 1670 records from staging


In [2]:
df

,id,source_id,drug_name,brand_name,manufacturer,status,reason,presentation,shortage_start,shortage_end,source,raw_json,ingested_at,therapeutic_category
0,1,,QUINAPRIL HYDROCHLORIDE TABLET,QUINAPRIL,"Solco Healthcare US, LLC",CURRENT,Shortage of an active ingredient,"Quinapril Hydrochloride, Tablet, 10 mg (NDC 43...",01/19/2023,04/29/2026,FDA,"{""update_type"": ""Reverified"", ""initial_posting...",2026-06-05 05:01:30.087,Cardiovascular
1,2,,EVEROLIMUS TABLET,EVEROLIMUS,"Teva Pharmaceuticals USA, Inc.",TO BE DISCONTINUED,,"Everolimus, Tablet, 7.5 mg (NDC 0093-7768-24)",03/02/2026,03/02/2026,FDA,"{""discontinued_date"": ""03/02/2026"", ""update_ty...",2026-06-05 05:01:30.087,Transplant
2,3,,PILOCARPINE HYDROCHLORIDE TABLET,PILOCARPINE HYDROCHLORIDE,"Actavis Pharma, Inc.",TO BE DISCONTINUED,,"Salagen, Tablet, 7.5 mg (NDC 0228-2837-11)",09/19/2025,09/19/2025,FDA,"{""discontinued_date"": ""09/19/2025"", ""update_ty...",2026-06-05 05:01:30.087,Other
3,4,,ETOMIDATE INJECTION,AMIDATE,"Hospira, Inc., a Pfizer Company",CURRENT,Other,"Amidate, Injection, 2 mg/mL (NDC 0409-6695-02)",10/05/2022,05/22/2026,FDA,"{""update_type"": ""Revised"", ""initial_posting_da...",2026-06-05 05:01:30.087,Anesthesia
4,5,,OMEPRAZOLE DELAYED RELEASE CAPSULE,OMEPRAZOLE,Sandoz Inc.,TO BE DISCONTINUED,,"Omeprazole, Delayed Release Capsule, 10 mg (ND...",10/09/2025,10/09/2025,FDA,"{""discontinued_date"": ""10/09/2025"", ""update_ty...",2026-06-05 05:01:30.087,Gastroenterology
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1665,1666,,FINASTERIDE TABLET,FINASTERIDE,"Teva Pharmaceuticals USA, Inc.",TO BE DISCONTINUED,,"Finasteride, Tablet, 5 mg (NDC 0093-7355-05)",08/19/2025,08/19/2025,FDA,"{""discontinued_date"": ""08/19/2025"", ""update_ty...",2026-06-05 05:01:30.117,Urology
1666,1667,,RAMELTEON TABLET,ROZEREM,Takeda Pharmaceuticals USA Inc.,TO BE DISCONTINUED,,"Rozerem, Tablet, 8 mg (NDC 64764-805-30)",03/19/2026,03/19/2026,FDA,"{""discontinued_date"": ""03/19/2026"", ""update_ty...",2026-06-05 05:01:30.117,Psychiatry
1667,1668,,"FLUVASTATIN SODIUM TABLET, EXTENDED RELEASE",LESCOL XL,Sandoz Inc.,TO BE DISCONTINUED,,"Lescol XL, Tablet, Extended Release, 80 mg (ND...",01/27/2026,01/27/2026,FDA,"{""discontinued_date"": ""01/27/2026"", ""update_ty...",2026-06-05 05:01:30.117,Cardiovascular
1668,1669,,"AMPHETAMINE ASPARTATE MONOHYDRATE, AMPHETAMINE...","DEXTROAMPHETAMINE SACCHARATE, AMPHETAMINE ASPA...","Teva Pharmaceuticals USA, Inc.",CURRENT,Other,"Amphetamine Aspartate Monohydrate, Amphetamine...",10/12/2022,06/02/2026,FDA,"{""update_type"": ""Revised"", ""initial_posting_da...",2026-06-05 05:01:30.117,Psychiatry


In [3]:
# CELL 2 — Populate dim_drug (FIXED — picks up real therapeutic_category)
def upsert_drugs(df, engine):
    drugs = df[["drug_name", "therapeutic_category"]].drop_duplicates().dropna(subset=["drug_name"])
    drugs = drugs[drugs["drug_name"] != ""].copy()
    drugs.columns = ["generic_name", "therapeutic_category"]
    drugs["normalized_name"] = drugs["generic_name"].str.upper().str.strip()
    drugs["therapeutic_category"] = drugs["therapeutic_category"].fillna("UNCATEGORIZED")
    drugs["is_essential"] = False
    drugs["criticality_tier"] = "MEDIUM"
    drugs["risk_score"] = 0.0

    existing = pd.read_sql("SELECT normalized_name FROM core.dim_drug", engine)["normalized_name"].tolist()
    new_drugs = drugs[~drugs["normalized_name"].isin(existing)]

    if not new_drugs.empty:
        new_drugs.to_sql("dim_drug", engine, schema="core", if_exists="append", index=False)
        print(f"✅ Inserted {len(new_drugs)} new drugs into dim_drug")
    else:
        print("No new drugs to insert")

upsert_drugs(df, engine)



✅ Inserted 255 new drugs into dim_drug


In [4]:
# CELL — Update dim_drug with real therapeutic categories from staging

# Get the real categories from staging
staging_cats = pd.read_sql("""
    SELECT 
        UPPER(TRIM(drug_name)) as drug_name,
        therapeutic_category
    FROM staging.stg_fda_shortages
    WHERE therapeutic_category IS NOT NULL
        AND therapeutic_category != ''
""", engine)

print(f"Staging records with categories: {len(staging_cats)}")
print(staging_cats.head())

Staging records with categories: 1670
                            drug_name therapeutic_category
0      QUINAPRIL HYDROCHLORIDE TABLET       Cardiovascular
1                   EVEROLIMUS TABLET           Transplant
2    PILOCARPINE HYDROCHLORIDE TABLET                Other
3                 ETOMIDATE INJECTION           Anesthesia
4  OMEPRAZOLE DELAYED RELEASE CAPSULE     Gastroenterology


In [5]:
# CELL 3 — Populate dim_manufacturer
def upsert_manufacturers(df, engine):
    mfrs = df[["manufacturer"]].drop_duplicates().dropna()
    mfrs = mfrs[mfrs["manufacturer"] != ""]
    mfrs.columns = ["company_name"]
    mfrs["country_of_origin"] = "US"
    mfrs["is_sole_supplier"] = False

    existing = pd.read_sql("SELECT company_name FROM core.dim_manufacturer", engine)["company_name"].tolist()
    new_mfrs = mfrs[~mfrs["company_name"].isin(existing)]

    if not new_mfrs.empty:
        new_mfrs.to_sql("dim_manufacturer", engine, schema="core", if_exists="append", index=False)
        print(f"✅ Inserted {len(new_mfrs)} manufacturers")

upsert_manufacturers(df, engine)

✅ Inserted 135 manufacturers


In [6]:
# CELL 4 — Load fact_shortage
def load_facts(df, engine):
    drug_map = pd.read_sql("SELECT drug_id, normalized_name FROM core.dim_drug", engine)
    drug_map = dict(zip(drug_map["normalized_name"], drug_map["drug_id"]))

    mfr_map = pd.read_sql("SELECT manufacturer_id, company_name FROM core.dim_manufacturer", engine)
    mfr_map = dict(zip(mfr_map["company_name"], mfr_map["manufacturer_id"]))

    region_us = pd.read_sql("SELECT region_id FROM core.dim_region WHERE country_code='US'", engine)["region_id"][0]
    source_fda = pd.read_sql("SELECT source_id FROM core.dim_source WHERE source_name='FDA'", engine)["source_id"][0]

    rows = []
    for _, r in df.iterrows():
    
        drug_key = str(r["drug_name"]).upper().strip()
        mfr_key = str(r["manufacturer"])
    
        hash_key = hashlib.sha256(
            f"{drug_key}|{mfr_key}|{r['shortage_start']}|FDA".encode()
        ).hexdigest()
    
        # FDA → Warehouse status mapping
        fda_status = str(r["status"]).upper()
    
        if fda_status == "CURRENT":
            status = "ACTIVE"
        elif fda_status == "TO BE DISCONTINUED":
            status = "DISCONTINUED"
        elif fda_status == "RESOLVED":
            status = "RESOLVED"
        else:
            status = "UNKNOWN"
        print("Raw status:", r["status"])
        print("Mapped status:", status)
        
        rows.append({
            "drug_id":             drug_map.get(drug_key),
            "manufacturer_id":     mfr_map.get(mfr_key),
            "region_id":           region_us,
            "source_id":           source_fda,
            "source_record_id":    r["source_id"],
            "status":              status,
            "reason_detail":       r["reason"],
            "presentation":        r["presentation"],
            "shortage_start_date": pd.to_datetime(r["shortage_start"], errors="coerce"),
            "shortage_end_date":   pd.to_datetime(r["shortage_end"], errors="coerce"),
            "is_critical":         False,
            "is_resolved":         status == "RESOLVED",
            "record_hash":         hash_key,
            "ingested_at":         datetime.now(),
        })

    facts_df = pd.DataFrame(rows).dropna(subset=["drug_id"])

    existing_hashes = pd.read_sql("SELECT record_hash FROM core.fact_shortage", engine)["record_hash"].tolist()
    facts_df = facts_df[~facts_df["record_hash"].isin(existing_hashes)]
    print(facts_df[["shortage_start_date","shortage_end_date"]].head())
    print(facts_df.dtypes)
    if not facts_df.empty:
        facts_df.to_sql("fact_shortage", engine, schema="core", if_exists="append", index=False)
        print(f"✅ Loaded {len(facts_df)} records into fact_shortage")
    else:
        print("No new records to load")

load_facts(df, engine)


Raw status: CURRENT
Mapped status: ACTIVE
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: RESOLVED
Mapped status: RESOLVED
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: CURRENT
Mapped status: ACTIVE
Raw status: TO BE DISCONTINUED
Mapped status: DISCONTINUED
Raw status: CURRENT
Mapped status: ACTIVE
Raw 

In [7]:
df["presentation"].str.len().max()

325

In [8]:
df["status"].value_counts()

status
CURRENT               1144
TO BE DISCONTINUED     497
RESOLVED                29
Name: count, dtype: int64

In [9]:
df.loc[
    df["presentation"].str.len().idxmax(),
    "presentation"
]

'PROSOL 20% SULFITE FREE IN PLASTIC CONTAINER, Injection, 2.76 g/100 mL; 1.96 g/100 mL; 600 mg/100 mL; 1.02 g/100 mL; 2.06 g/100 mL; 1.18 g/100 mL; 1.08 g/100 mL; 1.08 g/100 mL; 1.35 g/100 mL; 760 mg/100 mL; 1 g/100 mL; 1.34 g/100 mL; 1.02 g/100 mL; 980 mg/100 mL; 320 mg/100 mL; 50 mg/100 mL; 1.44 g/100 mL (NDC 0338-0499-06)'

In [10]:
# CELL 5 — Final verification
summary = pd.read_sql("""
    SELECT
        COUNT(*) as total_shortages,
        COUNT(DISTINCT drug_id) as unique_drugs,
        COUNT(DISTINCT manufacturer_id) as manufacturers,
        SUM(CASE WHEN status='ACTIVE' THEN 1 ELSE 0 END) as active,
        SUM(CASE WHEN status='RESOLVED' THEN 1 ELSE 0 END) as resolved
    FROM core.fact_shortage
""", engine)
print("\n📊 DATABASE SUMMARY:")
print(summary.to_string())


📊 DATABASE SUMMARY:
   total_shortages  unique_drugs  manufacturers  active  resolved
0             1670           247            135    1144        29


In [11]:
upsert_drugs(df, engine)

No new drugs to insert
